# Lecture 2: Euler, Lagrange and the first variation

**The challenge.** Johann Bernoulli solved the brachistochrone with a trick:
light, Snell's law, layers of glass. It was brilliant and it was useless for
the next problem. Change the cost to the time plus the wire's length, or let
the bead feel air drag, and the optical analogy is gone. What is needed is a
*method*: a rule that, for any cost written as an integral, produces an
equation the optimal curve must satisfy. And for control problems the curve
is not free: it is tied to the controls by differential equations, which act
as *constraints* along the whole trajectory.

This lecture builds that method, the **first variation**, in two steps:
first for a free curve (the Euler–Lagrange equation), then for a curve
constrained by dynamics (Lagrange multipliers that become functions of
time, the **costates**). The second step is where every quantity the package
derives comes from.

**After this lecture you can**

- derive the Euler–Lagrange equation and its natural boundary conditions;
- adjoin dynamics with time-varying multipliers and read off the costate
  equations, the stationarity condition and the transversality conditions;
- find each of them in what `derive` returns.

## 1. A young man's letter

The first general method was Leonhard Euler's. In his *Methodus inveniendi
lineas curvas* (1744) he replaced the unknown curve by a polygon, varied one
vertex at a time, and let the number of vertices grow. Today we would call
that a *direct* method: discretize first, then optimize. It worked, and it
gave the equation that now bears his name. But every new problem needed a
new picture.

In August 1755 Euler, then 48 and the most famous mathematician in Europe,
received a letter from Turin. Its author, Joseph-Louis Lagrange, was 19. He
had found a way to do without the polygons. Vary the *whole* curve at once
by a small function $\delta y(t)$, expand the cost to first order, and
integrate by parts. Euler's equation then falls out of pure analysis, for
any integrand, with the boundary conditions as a bonus. Euler saw
immediately that it was better than his own approach. He held back his
own work on the method so that the young man could publish first, and he
named the subject the *calculus of variations* in his 1766 *Elementa calculi
variationum*. That same year, when Euler left Berlin for St Petersburg,
d'Alembert recommended Lagrange to Frederick the Great as his successor. Euler by then was going blind; he lost
the sight of one eye in 1738 and nearly all of it in 1771. About half of
his enormous output was dictated after that.

## 2. The first variation of a free curve

Take a cost

$$
J[y] = \int_{t_0}^{t_f} F\big(t, y(t), \dot y(t)\big)\,dt
$$

and perturb an optimal $y$ by $\varepsilon\,\eta(t)$. If $y$ is optimal,
$J[y + \varepsilon\eta]$ has a minimum at $\varepsilon = 0$, so its derivative
there, the **first variation**, vanishes for every admissible $\eta$:

$$
\delta J = \int_{t_0}^{t_f} \big(F_y\,\eta + F_{\dot y}\,\dot\eta\big)\,dt
= \int_{t_0}^{t_f} \Big(F_y - \frac{d}{dt} F_{\dot y}\Big)\eta\,dt
+ \Big[F_{\dot y}\,\eta\Big]_{t_0}^{t_f} = 0 .
$$

The second form is Lagrange's integration by parts. If the endpoints are
fixed, $\eta$ vanishes there, the bracket drops out, and since $\eta$ is
otherwise arbitrary (the *fundamental lemma* of the calculus of variations)
the integrand must vanish:

$$
\boxed{\;F_y - \frac{d}{dt}F_{\dot y} = 0\;} \qquad\text{(Euler–Lagrange)} .
$$

If an endpoint is **free**, $\eta$ need not vanish there, so the bracket must
vanish on its own: $F_{\dot y} = 0$ at that end. This is a **natural boundary
condition**. Nobody imposed it; optimality produced it.

SymPy can do the algebra. For the brachistochrone written as a curve
$y(x)$, the time is $\int \sqrt{1 + y'^2}\,/\sqrt{2 g y}\;dx$.

In [ ]:
# What a variation is: the optimal curve, nudged by eps * eta, and the cost as a function
# of eps. Here the curve is the brachistochrone y(x) and the cost its descent time.
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import brentq

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"
G = 9.81


def descent_time(xs, ys):
    """Frictionless descent time along a polyline (y down), from rest."""
    t, v = 0.0, 0.0
    for dx, dy in zip(np.diff(xs), np.diff(ys)):
        v1 = np.sqrt(max(v * v + 2 * G * dy, 1e-300))
        t += 2 * np.hypot(dx, dy) / (v + v1)
        v = v1
    return t


def cycloid(x_end, phi_end, n=4001):
    R = x_end / (phi_end - np.sin(phi_end))
    # Cluster points near the start, where the curve is vertical.
    phi = phi_end * np.linspace(0, 1, n) ** 2
    return R * (phi - np.sin(phi)), R * (1 - np.cos(phi))


XF, YF = 2.0, 1.0
phi_f = brentq(lambda f: (f - np.sin(f)) / (1 - np.cos(f)) - XF / YF, 1e-6, 2 * np.pi - 1e-6)
xs, ys = cycloid(XF, phi_f)
eta = np.sin(np.pi * xs / XF) * 0.25                 # vanishes at both ends

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.9))
ax = axes[0]
for e in (-1.0, -0.5, 0.5, 1.0):
    ax.plot(xs, ys + e * eta, color=GRAY, lw=1, alpha=0.8)
ax.plot(xs, ys + 1.0 * eta, color=ORANGE, lw=1.5, label=r"$y + \varepsilon\eta$, $\varepsilon = 1$")
ax.plot(xs, ys, color=BLUE, lw=2.5, label="optimal y (the cycloid)")
ax.fill_between(xs, ys, ys + eta, color=ORANGE, alpha=0.12, lw=0)
ax.text(1.0, ys[np.searchsorted(xs, 1.0)] + 0.17, r"$\varepsilon\eta(x)$", color=ORANGE, fontsize=10)
ax.plot([0, XF], [0, YF], "ko", ms=5)
ax.set(xlabel="x", ylabel="depth y", ylim=(1.45, -0.15))
ax.set_title(r"fixed ends: $\eta(0) = \eta(x_f) = 0$", fontsize=10, loc="left")
ax.legend(frameon=False, fontsize=8, loc="upper right")

ax = axes[1]
eps = np.linspace(-1, 1, 41)
T = np.array([descent_time(xs, ys + e * eta) for e in eps])
T0 = descent_time(xs, ys)
ax.plot(eps, T, color=BLUE, lw=2)
ax.plot(eps, T0 + 0 * eps, color=GRAY, lw=1, ls="--")
ax.plot(0, T0, "o", color=ORANGE, ms=8)
ax.annotate("zero slope at " r"$\varepsilon = 0$:" "\nthe first variation vanishes", xy=(0, T0),
            xytext=(-0.55, T0 + 0.012), fontsize=8.5, arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 0.8})
ax.set(xlabel=r"$\varepsilon$", ylabel=r"descent time $J[y + \varepsilon\eta]$ (s)")
ax.set_title("cost along the family", fontsize=10, loc="left")

ax = axes[2]
xs2, ys2 = cycloid(XF, np.pi)                        # the free-end optimum arrives horizontally
eta2 = (xs2 / XF) ** 1.5 * 0.3                       # zero at the start only
for e in (-1.0, -0.5, 0.5, 1.0):
    ax.plot(xs2, ys2 + e * eta2, color=GRAY, lw=1, alpha=0.8)
ax.plot(xs2, ys2, color=BLUE, lw=2.5)
ax.plot(0, 0, "ko", ms=5)
ax.axvline(XF, color="k", lw=1, ls=":")
ax.annotate("arrives level:\n" r"$F_{y'} = 0$, so $y' = 0$", xy=(XF, ys2[-1]), xytext=(0.95, 0.75),
            fontsize=9, arrowprops={"arrowstyle": "-|>", "color": "k", "lw": 0.8})
ax.text(XF - 0.04, -0.08, r"end anywhere on $x = x_f$", fontsize=8.5, ha="right")
ax.set(xlabel="x", ylabel="depth y", ylim=(1.75, -0.15))
ax.set_title(r"free end: $\eta(x_f) \ne 0$", fontsize=10, loc="left")
for a_ in axes:
    a_.grid(alpha=0.25)
fig.tight_layout()

*A variation, drawn. Left: the brachistochrone and a family $y + \varepsilon\eta$ that keeps both ends fixed. Middle: the descent time along that family, computed by quadrature, is smallest at $\varepsilon = 0$ and flat there. That flatness, for every $\eta$, is $\delta J = 0$. Right: if the end may slide along the line $x = x_f$, $\eta(x_f)$ need not vanish, and optimality adds the condition $F_{y'} = 0$ there: the fastest curve to a vertical line arrives horizontally.*

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from sympy.calculus.euler import euler_equations

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8a85"

s = sp.Symbol("s", real=True)                 # the horizontal coordinate
gs = sp.Symbol("g", positive=True)
Y = sp.Function("y")
F = sp.sqrt(1 + Y(s).diff(s) ** 2) / sp.sqrt(2 * gs * Y(s))
euler_equations(F, Y(s), s)[0]

That second-order equation is unpleasant, but $F$ does not depend on $s$, and
then the Euler–Lagrange equation has a first integral, the **Beltrami
identity** $F - y' F_{y'} = \text{const}$:

In [ ]:
yp = Y(s).diff(s)
beltrami = sp.simplify(F - yp * F.diff(yp))
beltrami

So $y(1 + y'^2)$ is constant along the optimal curve, say $2R$. The cycloid
$x = R(\varphi - \sin\varphi)$, $y = R(1 - \cos\varphi)$ satisfies exactly
that:

In [ ]:
phi, R = sp.symbols("varphi R", positive=True)
xc, yc = R * (phi - sp.sin(phi)), R * (1 - sp.cos(phi))
slope = sp.diff(yc, phi) / sp.diff(xc, phi)
sp.simplify(yc * (1 + slope**2))

Bernoulli's optics and Lagrange's analysis agree. The analysis, though, works
for any $F$.

## 3. Adding dynamics: multipliers that are functions of time

A control problem is not a free curve. The states are tied to the controls
by $\dot x = f(t, x, u)$ at *every* instant. In ordinary calculus a constraint
$c(x) = 0$ is handled by a Lagrange multiplier: minimize $F + \lambda c$
instead of $F$. Here there is one constraint per state *per instant*, so the
multiplier becomes a function of time, $\lambda(t) \in \mathbb{R}^n$. Adjoin
the dynamics to the Bolza cost of Lecture 1, and the terminal constraints
$\psi = 0$ with constant multipliers $\nu$:

$$
\bar J = \phi + \nu^\top \psi + \int_{t_0}^{t_f}
\Big[ L + \lambda^\top \big(f - \dot x\big) \Big] dt .
$$

Define the **Hamiltonian**

$$
H(t, x, \lambda, u) = L(t, x, u) + \lambda^\top f(t, x, u),
$$

integrate $-\lambda^\top\dot x$ by parts, and take the first variation with
respect to $x$, $u$ and the endpoint. For a fixed final time:

$$
\delta\bar J = \Big[\big(\phi_x + \nu^\top\psi_x - \lambda^\top\big)\delta x\Big]_{t_f}
+ \lambda^\top(t_0)\,\delta x(t_0)
+ \int_{t_0}^{t_f} \Big[ \big(H_x + \dot\lambda^\top\big)\delta x + H_u\,\delta u \Big] dt .
$$

Every term must vanish separately, because $\delta u$ and $\delta x$ are
arbitrary once $\lambda$ is chosen to kill the $\delta x$ terms. That gives
the necessary conditions:

| condition | equation | name |
|---|---|---|
| choice of $\lambda$ | $\dot\lambda = -H_x^\top$ | **costate equations** |
| arbitrary $\delta u$ | $H_u = 0$ | **stationarity** (the control law) |
| free $x_i(t_f)$ | $\lambda(t_f) = \phi_x^\top + \psi_x^\top \nu$ | **transversality** |
| free $x_i(t_0)$ | $\lambda_i(t_0) = 0$ | natural boundary condition |

With a free final time there is one more, from varying $t_f$:
$H(t_f) = -\partial(\phi + \nu^\top\psi)/\partial t_f$, which is
$H(t_f) = 0$ when neither depends on $t_f$ explicitly. Lecture 3 returns to it.

The states satisfy $2n$ boundary conditions in total, half at each end, and
$\dot x = f$, $\dot\lambda = -H_x$ is a system of $2n$ differential equations.
That is a **two-point boundary value problem**, and solving it is what the
rest of the course is about.

## 4. The same derivation, done by the package

Take the simplest problem with every ingredient:

$$
\min\; \tfrac{w}{2}\,x(T)^2 + \int_0^T \tfrac12 u^2\,dt,
\qquad \dot x = u,\quad x(0) = a,\quad x(T) \text{ free} .
$$

By hand: $H = \tfrac12 u^2 + \lambda u$, so $\dot\lambda = -H_x = 0$ (the
costate is constant), $H_u = u + \lambda = 0$ gives $u = -\lambda$, and
transversality gives $\lambda(T) = \phi_x = w\,x(T)$. Then
$x(T) = a - \lambda T$, so $\lambda = w a / (1 + w T)$. It is in the examples
as `soft_terminal_lq`. `derive` applies the table above:

In [ ]:
from auto_optimal_control import examples
from auto_optimal_control.pontryagin import derive

lq = derive(examples.soft_terminal_lq())
print("costates:        ", lq.costates)
print("H (controls free):", lq.hamiltonian_full)
print("control law:     ", lq.control_law)
print("H at the law:    ", lq.hamiltonian)
print("costate dynamics:", lq.costate_dynamics)

The boundary conditions are collected in `lq.boundary`. They are written in
the values at both ends: `x(t_0)`, `lam_x(t_f)` and so on, plus the terminal
multipliers `nu` (none here, since $x(T)$ is free).

In [ ]:
b = lq.boundary
print("unknowns:       ", b.unknowns)
print("initial:        ", b.initial)
print("transversality: ", b.transversality)

Transversality reads $\lambda(t_f) - w\,x(t_f) = 0$, exactly as derived.
Solve the boundary value problem (again as a black box for now) and compare
with $\lambda = w a/(1 + wT)$:

In [ ]:
from auto_optimal_control.homotopy import Homotopy

T, w, a = 2.0, 3.0, 1.0
r = Homotopy(examples.soft_terminal_lq()).solve([T, w, a], ([-5], [5]))
print(f"lambda(0) = {r.solution.lam0[0]:.15f}")
print(f"closed    = {w * a / (1 + w * T):.15f}")

### A free initial state

Now let the start be free instead. Minimize
$\int_0^1 \tfrac12(u^2 + x^2)\,dt$ with $\dot x = u$ and $x(1) = 1$, with
$x(0)$ anything. The natural boundary condition $\lambda(0) = 0$ replaces
the missing initial condition. By hand: $H = \tfrac12(u^2 + x^2) + \lambda u$,
$u = -\lambda$, $\dot\lambda = -x$, so $\ddot x = x$ with $\dot x(0) = 0$ and
$x(1) = 1$, which gives $x = \cosh t / \cosh 1$.

In [ ]:
from auto_optimal_control.problem import OptimalControlProblem

t, x, u = sp.symbols("t x u", real=True)
free_start = OptimalControlProblem(
    name="free_start", time=t, states=[x], controls=[u],
    dynamics=[u], running_cost=(u**2 + x**2) / 2,
    final_state={x: 1}, final_time=1,           # no initial_state: x(0) is free
)
fs = derive(free_start)
print("initial conditions:", fs.boundary.initial)

r = Homotopy(free_start).solve([], ([-5], [5]))
tr = r.shooting.trajectory(r.solution.v, n_points=101)
err = np.abs(tr["z"][:, 0] - np.cosh(tr["t"]) / np.cosh(1)).max()
print(f"x(0) = {r.solution.x0[0]:.12f} (closed form 1/cosh 1 = {1 / np.cosh(1):.12f})")
print(f"max |x(t) - cosh t / cosh 1| = {err:.1e}")

The package did not need to be told where $x$ starts: it imposed
$\lambda(0) = 0$ and found $x(0) = 1/\cosh 1$. Free initial states are
unknowns of the shooting (Lecture 9), with that natural condition as their
residual.

## 5. What the multiplier means

Lagrange multipliers in ordinary calculus measure how the optimal cost
changes when a constraint is relaxed. Costates do the same thing, state by
state and instant by instant: $\lambda(t)$ is the sensitivity of the optimal
remaining cost to the state at time $t$. Lecture 5 proves it with dynamic
programming. Here is a numerical preview. Nudge the initial state of the
soft-terminal problem, re-solve, and compare the change in cost with
$\lambda(0)$:

In [ ]:
h = Homotopy(examples.soft_terminal_lq())


def optimal_cost(a0):
    return h.solve([T, w, a0], ([-5], [5])).objective


d = 1e-5
dJ = (optimal_cost(a + d) - optimal_cost(a - d)) / (2 * d)
print(f"dJ*/dx0   = {dJ:.8f}")
print(f"lambda(0) = {w * a / (1 + w * T):.8f}")

They agree. The costate is a price: how much one more unit of state, at that
moment, would cost.

## Summary

- The first variation sets the derivative of the cost along every
  perturbation to zero. For a free curve it gives the Euler–Lagrange equation
  and natural boundary conditions at free ends.
- Dynamics are adjoined with multipliers $\lambda(t)$. With
  $H = L + \lambda^\top f$, the necessary conditions are $\dot\lambda = -H_x$,
  $H_u = 0$, transversality $\lambda(t_f) = \phi_x + \psi_x^\top \nu$, and
  $\lambda_i(t_0) = 0$ for free initial states.
- States and costates together obey a two-point boundary value problem.
- `derive` returns each piece: `hamiltonian_full`, `control_law`,
  `costate_dynamics` and `boundary` (`initial`, `transversality`, `unknowns`).
- The costate is the sensitivity of the optimal cost to the state.

## Exercises

1. Use `euler_equations` on $F = \sqrt{1 + y'^2}$ (arc length) and show the
   shortest path is a straight line.
2. For the soft-terminal problem, find $\lambda$ as $w \to \infty$. Which hard
   terminal condition does the problem approach? Check with the package for
   $w = 10^4$.
3. In the free-start problem, make the final state free as well and add a
   terminal cost $\tfrac12(x(1) - 1)^2$. Derive the boundary conditions by
   hand, then compare with `derive(...).boundary`.

## Further reading

- I. M. Gelfand and S. V. Fomin, *Calculus of Variations* (1963), chapters 1
  and 2: the first variation, done carefully.
- H. H. Goldstine, *A History of the Calculus of Variations from the 17th
  through the 19th Century* (1980), chapters 2 and 3: Euler and Lagrange.
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control* (1969), sections 2.1
  to 2.3.